# 01 - Data exploration

What the panel contains, and whether it can support the modelling plan.

> **These notebooks define no functions.** Everything they call lives in `src/`.
> That rule is from `brain.md` section 7: logic written in a cell cannot be tested
> and silently drifts from the module, which is how report figures stop matching
> the code that ships.
>
> Until real case data is in `data/raw/cases/`, these fall back to a generated
> panel and every number describes a generator rather than dengue.

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

from src.config import load_config

cfg = load_config("../config.yaml")
cfg.project.name, cfg.project.granularity

In [ ]:
from src.panel import assemble_panel
from src.preprocess import preprocess

try:
    panel = assemble_panel(cfg)
    SYNTHETIC = False
except Exception as error:
    print("no real data yet, using the synthetic stand-in:")
    print(" ", str(error).splitlines()[0])
    from src.synthetic import synthetic_panel
    panel = synthetic_panel(cfg)
    SYNTHETIC = True

clean = preprocess(panel, cfg).panel[list(panel.columns)]
clean.shape

## Does the data support the plan?

`periods_per_state` is the number that decides whether a per-state model is possible at all.

In [ ]:
from src.panel import summarise_panel

print(summarise_panel(clean, cfg).describe())

## Coverage, gaps and outliers

Per state and variable. `longest_missing_run` matters more than the count: scattered gaps interpolate, consecutive ones do not.

In [ ]:
from src.panel import data_quality_report

report = data_quality_report(clean, cfg)
report.sort_values("coverage").head(15)

## What preprocessing did

Case counts are never interpolated. A month without surveillance is not a month with an estimable number of cases.

In [ ]:
result = preprocess(panel, cfg)
print(result.describe())
result.long_gaps.head(10)

## Seasonality

The shape everything downstream depends on. If the season is not visible here, no model will find it.

In [ ]:
import matplotlib.pyplot as plt

monthly = (clean.reset_index()
           .assign(month=lambda f: f['date'].dt.month)
           .groupby('month')['cases'].mean())
ax = monthly.plot(figsize=(7, 2.6), color='#0F6E8C')
ax.set_ylabel('mean cases'); ax.set_xlabel('month')
ax.spines[['top', 'right']].set_visible(False)
plt.show()